# Agent

In [ ]:
# @title ### 1. SETUP & DEPENDENCIES
!pip install -q google-adk google-cloud-bigquery pandas

import os
import asyncio
import pandas as pd
import google.auth
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.tools.bigquery import BigQueryCredentialsConfig, BigQueryToolset
from google.adk.tools.bigquery.config import BigQueryToolConfig, WriteMode
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools.agent_tool import AgentTool

# --- 1. Environment & Project Config ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

PROJECT_ID = "project-nirvana-405904"
DATASET_ID = "vel_csv_schema"
MODEL_NAME = "gemini-2.5-flash"

# --- 2. Toolset Configuration ---
bq_tool_config = BigQueryToolConfig(write_mode=WriteMode.BLOCKED)
application_default_credentials, _ = google.auth.default()
bq_credentials_config = BigQueryCredentialsConfig(credentials=application_default_credentials)

bigquery_toolset = BigQueryToolset(
    credentials_config=bq_credentials_config,
    bigquery_tool_config=bq_tool_config
)

In [ ]:
# @title ### 2. AGENT DEFINITIONS (From Source Code)
SYSTEM_INSTRUCTION_BQ = f"""
==================================================
1. SYSTEM PERSONA & OBJECTIVES
==================================================
You are an advanced Strategic Business Analyst AI for the YouTube Partnerships team. Your primary objective is to analyze high-stakes 1:1 conversations between YouTube's top Creators and their Strategic Partner Managers (SPMs), as well as the business signals derived from them.

Your audience consists of two main groups:
1. YouTube/Google Leaders: Looking for macro-level trends, product feedback, recurring regional issues, and overall SPM performance.
2. Strategic Partner Managers (SPMs): Looking for micro-level insights, preparation for their next creator call, follow-up tracking, and personalized coaching feedback based on past interactions.

You must remain analytical, objective, and strictly grounded in the provided data.

==================================================
2. BUSINESS CONTEXT & ANALYTICAL VALUE
==================================================
The data you have access to represents real, confidential meetings where top creators discuss their channel performance, monetization, platform issues, and content strategies with YouTube SPMs.

Your analysis provides critical value by:
Identifying Product/Platform Friction: Highlighting recurring bugs, policy confusion, or feature requests directly from top creators.
Tracking Longitudinal Trends: By analyzing consecutive conversations between the same SPM and Creator over time, you must detect if an issue is persisting, if a creator's sentiment is improving/degrading, or if the SPM successfully closed the loop on a previous action item.
Evaluating SPM Effectiveness: Assessing how well the SPM handled objections, demonstrated empathy, and adhered to the agenda.

==================================================
3. DATA SOURCE 1: EXTRACTED SIGNALS & VALIDATION
==================================================
BigQuery Table: project-nirvana-405904.vel_csv_schema.vel_csv_derived_signals_006

Context: This table contains the organic business signals (risks, opportunities, product feedback) successfully extracted from the raw transcripts, mapped strictly to our official taxonomy. It also includes an evaluation of the SPM's effectiveness.

Important Data Rules:
1-to-Many Relationship: A single conversation (conversation_id) can generate multiple rows if multiple distinct signals were detected.
No Signals Detected: If a transcript has no organic signals matching the strict taxonomy, a single row is still created to preserve the SPM evaluation metadata, with signal fields set to NULL.
Evidence Grouping: Multiple verbatim quotes for the exact same signal topic within a single conversation are concatenated in the signal_evidence field, separated by a pipe (|).

Schema:
conversation_id (STRING): Unique identifier for the conversation.
channel_name (STRING): The name of the YouTube channel discussed.
creator_niche (STRING): The content vertical or niche of the creator.
creator_region (STRING): The geographical region of the creator.
spm_name (STRING): The name of the YouTube Strategic Partner Manager.
duration_minutes (FLOAT): The length of the conversation in minutes.
spm_score (INTEGER): Grade (0-100) on the SPM's effectiveness based on empathy, clarity, resolution, and agenda adherence.
spm_reasoning (STRING): Justification for the spm_score.
signal_category (STRING): The macro-classification of the issue (e.g., Monetization, Content & Formats, Tools and Policy).
signal_sub_category (STRING): The secondary classification level under the category.
signal_name (STRING): The specific core topic of the detected issue, strictly matching the official taxonomy list.
signal_sentiment (STRING): The creator's tone regarding the topic (Positive, Negative, or Neutral).
signal_actionability (STRING): Determines if the issue is actionable by YouTube/SPM ("Actionable") or out of their control ("Non-Actionable").
signal_description (STRING): A brief summary of the specific issue discussed.
signal_evidence (STRING): Verbatim quote(s) extracted directly from the transcript proving the signal's existence. Multiple quotes separated by |.
recommended_action (STRING): Suggested next steps for the SPM or YouTube to address the overarching signal(s).
processed_at (TIMESTAMP): When the AI processing and extraction occurred.
validation_signal_flag (STRING): A flag placeholder for future QA validation steps.

==================================================
4. DATA SOURCE 2: RAW TRANSCRIPTS
==================================================
BigQuery Table: project-nirvana-405904.vel_csv_schema.vel_csv_synthetic_transcripts_006

Context: This table contains the actual, chronological dialogue of the conversations.

Important Data Rules:
Quality Filter: YOU MUST ONLY USE records where is_valid = TRUE. Ignore any row where this is false, as it indicates a corrupted or invalid recording.
JSON Parsing: The raw_transcript field is a JSON array of objects. Each object represents a chronological turn in the conversation. The structure is [{{ "role": "SPM" or "Creator", "content": "The actual spoken text or stage direction" }}]. You
must parse this array chronologically to understand the flow of the conversation.

Schema (Allowed Fields Only):
conversation_id (STRING): Unique identifier to join with Data Source 1.
creator_id (STRING): Unique identifier for the creator.
channel_name (STRING): Name of the channel.
spm_id (STRING): Unique identifier for the SPM.
spm_name (STRING): Name of the SPM.
recording_start (TIMESTAMP): The exact time the conversation began. Use this to establish chronological order for trend analysis.
recording_end (TIMESTAMP): The exact time the conversation ended.
raw_transcript (JSON): The full conversation dialogue.
is_valid (BOOLEAN): Must be TRUE.

==================================================
5. EXECUTION & REASONING GUIDELINES
==================================================
Joining Data: Always use conversation_id to link the derived signals (Source 1) with the exact dialogue (Source 2).
Time-Series Analysis: When asked about a specific creator or SPM over time, order the conversations using the recording_start field. Evaluate if issues raised in an earlier date were addressed in a later date.
Evidence-Based Answers: Whenever you claim a signal exists or a creator is upset, you must back it up using either the signal_evidence from Source 1 or a direct quote parsed from the raw_transcript JSON in Source 2.

==================================================
6. USE CASES & IMPORTANT QUERIES (EXAMPLES)
==================================================
To guide your analysis, here are the primary use cases and the types of strong queries you must be prepared to answer accurately based on the data:

USE CASE A: MACRO-LEVEL TRENDS (For YouTube/Google Leaders)
Leaders will ask you to aggregate data, spot regional trends, and evaluate overall SPM performance.
Important Queries to expect:
"Summarize the top 3 'Non-Actionable' product complaints (signal_category = 'Tools and Policy') across all creators in the LATAM region over the last month."
"Which SPMs have the lowest average spm_score when dealing with 'Monetization' signals, and what is the primary spm_reasoning the AI provided for those low scores?"
"Are there any emerging negative signals in the 'Content & Formats' sub-category that multiple different creators have raised in the last two weeks?"

USE CASE B: MICRO-LEVEL INSIGHTS & CALL PREP (For SPMs)
SPMs will ask you to analyze specific creators, prep for upcoming 1:1s, and track issue resolution over time.
Important Queries to expect:
- Review the last 3 transcripts for channel_name 'X' using the recording_start dates. Did the creator's negative sentiment regarding 'Algorithm changes' improve in the most recent call?
- Based on the recommended_action from SPM Tricia's last call with creator Rohan, what specific follow-up questions should she prioritize in her meeting today?
- Find the exact quotes (signal_evidence or parsed from raw_transcript) where creator Y expressed frustration about Shorts monetization. I need the verbatim context.
- Analyze the latest transcript for Creator Z. Did the SPM successfully adhere to the agenda and demonstrate empathy? Reference the spm_reasoning and the raw dialogue to justify your answer.

==================================================
7. GOLDEN QUERY REFERENCE (FEW-SHOT EXAMPLES)
==================================================
Use the following queries as the standard for complex business logic.
DO NOT modify the logic for joins or window functions.

### PATTERN 1: SPM STRESS LOAD METRICS
**Context:** Use this when asked about SPM burnout, workload hostility/stress level or SPM load. You could also leverage this query and tweak to check for questions that suggest SPM with low performance by taking a user prompt and adjusting the threshold to identify what % of negative calls would be identified as low performance etc.
It identifies calls where >50% of the topics discussed were negative and aggregates this per SPM.

```sql
WITH ValidSignals AS (
    SELECT
        s.conversation_id,
        s.spm_name,
        s.signal_name,
        s.signal_sentiment,
        t.recording_start
    FROM `{PROJECT_ID}.{DATASET_ID}.vel_csv_derived_signals_006` s
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.vel_csv_synthetic_transcripts_006` t
        ON s.conversation_id = t.conversation_id
    WHERE t.is_valid = TRUE
      AND s.signal_name IS NOT NULL
      AND t.recording_start >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
),
CallLevelStress AS (
    SELECT
        conversation_id,
        spm_name,
        COUNT(signal_name) AS total_topics_discussed,
        SUM(CASE WHEN signal_sentiment = 'Negative' THEN 1 ELSE 0 END) AS negative_topics_count,
        CASE
            WHEN SUM(CASE WHEN signal_sentiment = 'Negative' THEN 1 ELSE 0 END) / COUNT(signal_name) > 0.5
            THEN 1
            ELSE 0
        END AS is_stressful_call
    FROM ValidSignals
    GROUP BY conversation_id, spm_name
)
SELECT
    spm_name,
    COUNT(conversation_id) AS total_calls_this_month,
    SUM(is_stressful_call) AS hostile_calls_faced,
    ROUND(SAFE_DIVIDE(SUM(is_stressful_call), COUNT(conversation_id)) * 100, 2) AS stress_load_percentage
FROM CallLevelStress
GROUP BY spm_name
HAVING total_calls_this_month > 0
ORDER BY stress_load_percentage DESC, hostile_calls_faced DESC;


PATTERN 2: DECLINING CREATOR EXPERIENCE OVER TIME
Context: Use this when asked about "unhappy creators trend," "sentiment trends," or "issue escalation."
It uses window functions (LAG) to track the trajectory of sentiment across the last 3 interactions for the same topic.

WITH UnifiedData AS (
    SELECT
        s.channel_name,
        s.spm_name,
        s.signal_name,
        s.signal_sentiment,
        t.recording_start,
        s.conversation_id,
        CASE
            WHEN s.signal_sentiment = 'Positive' THEN 3
            WHEN s.signal_sentiment = 'Neutral' THEN 2
            WHEN s.signal_sentiment = 'Negative' THEN 1
            ELSE NULL
        END AS sentiment_score
    FROM `{PROJECT_ID}.{DATASET_ID}.vel_csv_derived_signals_006` s
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.vel_csv_synthetic_transcripts_006` t
        ON s.conversation_id = t.conversation_id
    WHERE t.is_valid = TRUE
      AND s.signal_name IS NOT NULL
      AND s.signal_sentiment IS NOT NULL
),
ChronologicalTracking AS (
    SELECT
        channel_name,
        spm_name,
        signal_name,
        LAG(recording_start, 2) OVER(PARTITION BY channel_name, signal_name ORDER BY recording_start ASC) AS call_1_date,
        LAG(sentiment_score, 2) OVER(PARTITION BY channel_name, signal_name ORDER BY recording_start ASC) AS call_1_score,
        LAG(recording_start, 1) OVER(PARTITION BY channel_name, signal_name ORDER BY recording_start ASC) AS call_2_date,
        LAG(sentiment_score, 1) OVER(PARTITION BY channel_name, signal_name ORDER BY recording_start ASC) AS call_2_score,
        recording_start AS call_3_date,
        sentiment_score AS call_3_score,
        conversation_id AS call_3_conversation_id
    FROM UnifiedData
)
SELECT
    channel_name,
    spm_name,
    signal_name AS persistent_friction_topic,
    CASE
        WHEN call_1_score = 1 AND call_2_score = 1 AND call_3_score = 1 THEN 'Chronic Negativity'
        WHEN (call_1_score = 3 AND call_2_score = 1 AND call_3_score = 1) OR
             (call_1_score = 3 AND call_2_score = 3 AND call_3_score = 1) THEN 'Abrupt Decline'
        WHEN (call_1_score = 3 AND call_2_score = 2 AND call_3_score = 1) OR
             (call_1_score = 2 AND call_2_score = 1 AND call_3_score = 1) OR
             (call_1_score = 2 AND call_2_score = 2 AND call_3_score = 1) THEN 'Gradual Decline'
        ELSE 'Other/Review'
    END AS scenario_label,
    call_1_score, call_1_date, call_2_score, call_2_date, call_3_score, call_3_date,
    call_3_conversation_id AS intervention_target_conversation
FROM ChronologicalTracking
WHERE call_1_score IS NOT NULL
  AND call_3_score = 1
  AND ( (call_1_score >= call_2_score AND call_2_score >= call_3_score) )
ORDER BY call_3_date DESC, channel_name ASC;

PATTERN 2: Issue Resolution & Sentiment Recovery
Context: Use this when asked "Which SPMs are best at resolving issues?" or "Has the creator's sentiment on Topic X improved?"
It identifies instances where a signal_name transitioned from 'Negative' in a previous call to 'Positive' in the most recent call.
WITH SentimentFlow AS (
    -- Step 1: Map sentiment and get the previous status for the SAME topic/channel
    SELECT
        s.channel_name,
        s.spm_name,
        s.signal_name,
        s.signal_sentiment AS current_sentiment,
        LAG(s.signal_sentiment) OVER(
            PARTITION BY s.channel_name, s.signal_name
            ORDER BY t.recording_start ASC
        ) AS previous_sentiment,
        t.recording_start AS current_call_date,
        s.conversation_id
    FROM `{PROJECT_ID}.{DATASET_ID}.vel_csv_derived_signals_006` s
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.vel_csv_synthetic_transcripts_006` t
        ON s.conversation_id = t.conversation_id
    WHERE t.is_valid = TRUE
      AND s.signal_name IS NOT NULL
),
ResolutionEvents AS (
    -- Step 2: Define a "Resolution" as a move from Negative -> Positive
    SELECT
        *,
        CASE
            WHEN previous_sentiment = 'Negative' AND current_sentiment = 'Positive' THEN 1
            ELSE 0
        END AS is_resolution
    FROM SentimentFlow
)
-- Step 3: Aggregate by SPM to find your "Fixers"
SELECT
    spm_name,
    COUNT(*) AS total_topic_interactions,
    SUM(is_resolution) AS issues_resolved,
    ROUND(SAFE_DIVIDE(SUM(is_resolution), COUNT(*)) * 100, 2) AS resolution_rate_percentage
FROM ResolutionEvents
WHERE previous_sentiment IS NOT NULL
GROUP BY spm_name
ORDER BY issues_resolved DESC, resolution_rate_percentage DESC;
"""

# --- 3. THE SQL ANALYST (Your Extensive Instructions Preserved) ---
sql_analyst = Agent(
    name="BQ_Data_Specialist",
    model=MODEL_NAME,
    description="Deep-dive SQL analyst using official Project Nirvana schema.",
    instruction=SYSTEM_INSTRUCTION_BQ,
    tools=[bigquery_toolset],
)

# --- 4. THE WEB RESEARCHER (The Supplement Agent) ---
web_researcher = Agent(
    name="External_Context_Specialist",
    model=MODEL_NAME,
    description="Finds real-time external creator news and trends.",
    instruction="""
    You provide supplemental information from the live web.
    - Focus: Latest creator news, social media sentiment, new product launches (e.g., MrBeast's Feastables).
    - Goal: Provide context that is NOT in the internal BigQuery database.
    - Important: Do not make up internal meeting data. Only report web findings.
 **IMPORTANT GUARDRAIL**: if you don't find any relevant information from search for a particular creator or any YouTube product do not hallucinate and explicitly mention you don't have additional information from the websearch. )

    """,
    tools=[GoogleSearchTool(bypass_multi_tools_limit=True)],
)

# --- THE PRINCIPAL DISPATCHER (Lead Strategist) ---
STRATEGIST_INSTRUCTION = """
### ROLE: Lead Strategic Dispatcher
You are the brain of the Agentic System. Your job is not just to summarize, but to INTELLIGENTLY ROUTE and SYNTHESIZE.

### STEP 1: INTENT CLASSIFICATION
Before calling sub-agents/tools, determine the scope:
- **Internal Only:** (e.g., "What is the average SPM score?") -> Call ONLY sql_analyst.
- **External Context:** (e.g., "What are the latest YouTube policy trends?") -> Call web_researcher.

### STEP 2: STRUCTURED OUTPUT
Present your findings in this EXACT order:

### 📊 INTERNAL MEETING INSIGHTS (From BigQuery)
- Summarize call history and SPM performance.
- ALWAYS show the SQL used to query the user's prompt

### 🌐 MARKET CONTEXT (From Google Search)
- Present relevant news or public trends.

### 💡 STRATEGIC SYNTHESIS
- This is your most important value-add. Connect the dots.
- Example: "The creator is complaining about views in BQ, but the YouTube API shows 10% growth. This suggests a perceived performance gap that the SPM needs to address with data."

**GUARDRAIL:** If an agent returns 'No information', do not omit the section; state 'No internal/external data available for this specific query.'
"""

lead_strategist = Agent(
    name="Lead_Strategist",
    model=MODEL_NAME, # Use Pro for high-level coordination
    description="Principal Orchestrator.",
    instruction=STRATEGIST_INSTRUCTION,
    # sub_agents=[sql_analyst, web_researcher]  ## tooling limitation in ADK
    tools=[
        AgentTool(agent=web_researcher),
        AgentTool(agent=sql_analyst)
    ],
)

# Main Code

In [ ]:
# =====================================================================
# @title ### 1. EVALUATION SETUP & DEPENDENCIES
# =====================================================================
import os
import re
import json
import vertexai
from vertexai.generative_models import GenerativeModel
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

# --- 1. Judge Model Configuration (Vertex AI) ---
# Initialize Vertex AI automatically using Colab Enterprise default credentials
# It reuses the PROJECT_ID and location defined in your Agent Setup block.
vertexai.init(
    project=PROJECT_ID,
    location=os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
)

# Initialize the Judge Model natively through Vertex AI
judge_model = GenerativeModel("gemini-2.5-pro")

# --- 2. Attach Existing Agent to Runner ---
# Initialize ADK Services using the 'lead_strategist' defined in the Agent block
session_service = InMemorySessionService()
runner = Runner(
    app_name="batch_eval_suite",
    agent=lead_strategist,
    session_service=session_service,
    auto_create_session=True # To create session where the evaluation will be saved
)

In [ ]:
# =====================================================================
# @title ### 2. Test Dataset Definition
# =====================================================================
# List of 15 business queries mapped to the specified BQ tables
test_dataset = [
    {
        "id": 1,
        "question": "What is the overall average sentiment score extracted from vel_csv_derived_signals_006?",
        "ground_truth": "Query should target the 'sentiment_score' column in vel_csv_derived_signals_006 and return an aggregated average."
    },
    {
        "id": 2,
        "question": "Summarize the key pain points from the transcript associated with call_id 'C-1001' in vel_csv_synthetic_transcripts_006.",
        "ground_truth": "Agent must query vel_csv_synthetic_transcripts_006 for call_id 'C-1001' and provide a natural language summary of the text."
    },
    {
        "id": 3,
        "question": "How many derived signals in vel_csv_derived_signals_006 have a 'high_risk' flag set to True?",
        "ground_truth": "Query must perform a COUNT operation filtered by 'high_risk' = True on the signals table."
    },
    {
        "id": 4,
        "question": "Cross-reference the main topic from transcript 'C-1002' with current Google Search trends.",
        "ground_truth": "Agent should fetch transcript C-1002, identify the topic, and execute a Google Search query for current trends."
    },
    {
        "id": 5,
        "question": "Identify the top 5 longest calls from vel_csv_synthetic_transcripts_006 and list their associated agent IDs.",
        "ground_truth": "Query must order by call duration descending, limit to 5, and select agent_id."
    },
    {
        "id": 6,
        "question": "Is there a correlation between negative sentiment in vel_csv_derived_signals_006 and mention of the word 'refund' in transcripts?",
        "ground_truth": "Agent needs to perform a JOIN between both tables on call_id, filtering for 'refund' (LIKE '%refund%') and comparing sentiment scores."
    },
    {
        "id": 7,
        "question": "Search Google for the latest competitor product launch and see if any transcripts mention this product name.",
        "ground_truth": "Must use GoogleSearchTool first to find the product name, then use BigQueryToolset to search transcripts for that string."
    },
    {
        "id": 8,
        "question": "Calculate the percentage of calls categorized as 'technical issue' in the signals table.",
        "ground_truth": "Mathematical calculation based on count of 'technical issue' category divided by total rows in vel_csv_derived_signals_006."
    },
    {
        "id": 9,
        "question": "Provide a strategic synthesis of customer satisfaction for Q3 based on the derived signals table.",
        "ground_truth": "Query filtered by Q3 dates, aggregating satisfaction signals, followed by a strategic textual analysis."
    },
    {
        "id": 10,
        "question": "Find the transcript with the lowest sentiment score and summarize the customer's main complaint.",
        "ground_truth": "Requires joining signals and transcripts on call_id, ordering by sentiment ASC limit 1, and summarizing."
    },
    {
        "id": 11,
        "question": "List all distinct signal categories identified in vel_csv_derived_signals_006.",
        "ground_truth": "Query must use SELECT DISTINCT on the category column of the signals table."
    },
    {
        "id": 12,
        "question": "What is the average response time for calls flagged with 'escalation' in the signals table?",
        "ground_truth": "Query filtering for 'escalation' flag and averaging the response time column."
    },
    {
        "id": 13,
        "question": "Find transcripts containing the phrase 'cancel subscription' and retrieve their corresponding risk signals.",
        "ground_truth": "Join both tables, filter transcript text for 'cancel subscription', and select risk columns from signals."
    },
    {
        "id": 14,
        "question": "Using Google Search, find standard call center benchmark metrics and compare our average call duration to them.",
        "ground_truth": "Fetch industry benchmark via Search, query average duration from transcripts, and compare both."
    },
    {
        "id": 15,
        "question": "Generate a weekly executive summary using the most frequent signals and the general tone of the transcripts.",
        "ground_truth": "Requires multi-step analysis: aggregate top signals, sample transcript tones, and synthesize an executive report."
    }
]

In [ ]:
# =====================================================================
# @title ### 3. Evaluator (Judge) Logic
# =====================================================================
async def evaluate_response(question: str, ground_truth: str, agent_response: str) -> dict:
    """Uses a Gemini Judge to evaluate the agent's response against the ground truth."""

    evaluation_prompt = f"""
    You are an expert impartial judge evaluating an AI Agent's response.

    Question Asked: {question}
    Expected Behavior/Data (Ground Truth): {ground_truth}
    Actual Agent Response: {agent_response}

    Please rate the Actual Agent Response on a scale of 1 to 5 for the following metrics:
    1. Faithfulness: Did the agent use the correct data/SQL from BigQuery as requested? (1=Completely fabricated/wrong tool, 5=Perfect tool use and accurate data representation)
    2. Completeness: Did the agent address all parts of the strategic synthesis and question? (1=Missing major parts, 5=Comprehensive and complete)

    Return ONLY a valid JSON object with no markdown formatting, using this exact schema:
    {{
        "faithfulness_score": <int>,
        "completeness_score": <int>,
        "reasoning": "<string explaining the scores>"
    }}
    """

    try:
        # Run judge synchronously in an async wrapper or use genai async client if available
        # We use run_in_executor to avoid blocking the event loop with synchronous SDK calls
        loop = asyncio.get_running_loop()
        judge_result = await loop.run_in_executor(
            None,
            lambda: judge_model.generate_content(evaluation_prompt)
        )

        # Clean the response to ensure valid JSON (remove markdown blocks if the model adds them)
        raw_text = judge_result.text.strip()
        json_str = re.sub(r'```json\n|\n```|```', '', raw_text).strip()

        return json.loads(json_str)
    except Exception as e:
        print(f"Error evaluating response: {e}")
        return {
            "faithfulness_score": 0,
            "completeness_score": 0,
            "reasoning": f"Evaluation failed: {str(e)}"
        }

In [ ]:
# =====================================================================
# @title ### 4. Main Execution Loop
# =====================================================================
async def run_test_case(test_case: dict, semaphore: asyncio.Semaphore) -> dict:
    """Executes a single test case through the ADK Runner and evaluates it."""
    async with semaphore:
        print(f"Executing Test Case {test_case['id']}...")

        # Create a unique session ID for isolation
        session_id = f"eval_session_{test_case['id']}"

        agent_response_text = ""

        try:
            # 1. Package the string into a valid ADK Content object to avoid 'str has no attribute role'
            try:
                user_message = genai_types.Content(
                    role="user",
                    parts=[{"text": test_case["question"]}]
                )
            except Exception:
                user_message = {"role": "user", "parts": [{"text": test_case["question"]}]}

            # 2. Execute the async stream
            async for event in runner.run_async(
                user_id="evaluator_bot",
                session_id=session_id,
                new_message=user_message
            ):
                # 3. Robustly extract text from the varying event stream
                chunk = None
                if hasattr(event, 'text') and event.text:
                    chunk = event.text
                elif hasattr(event, 'content') and event.content:
                    chunk = event.content
                elif hasattr(event, 'message') and hasattr(event.message, 'content'):
                    chunk = event.message.content

                # 4. Validate that it is a String before concatenating
                if chunk:
                    if isinstance(chunk, str):
                        agent_response_text += chunk
                    elif hasattr(chunk, 'parts'):
                        # If it is a Content object, extract text from its 'parts'
                        agent_response_text += "".join([p.text for p in chunk.parts if hasattr(p, 'text') and p.text])
                    else:
                        # Absolute safety fallback
                        agent_response_text += str(chunk)

            if not agent_response_text:
                agent_response_text = "Agent execution finished but returned empty text. Review Tool permissions."

        except Exception as e:
            agent_response_text = f"Agent execution failed: {str(e)}"

        # Call the Judge
        evaluation = await evaluate_response(
            question=test_case["question"],
            ground_truth=test_case["ground_truth"],
            agent_response=agent_response_text
        )

        return {
            "id": test_case["id"],
            "question": test_case["question"],
            "expected_behavior": test_case["ground_truth"],
            "agent_response": agent_response_text,
            "faithfulness": evaluation.get("faithfulness_score", 0),
            "completeness": evaluation.get("completeness_score", 0),
            "judge_reasoning": evaluation.get("reasoning", "No reasoning provided.")
        }

async def main(TEST_CASES_TO_RUN = -1):
    subset_to_run = test_dataset[:TEST_CASES_TO_RUN]
    print("Starting Batch Evaluation Suite...")

    # Use a semaphore to limit concurrent API calls and avoid Rate Limit errors
    semaphore = asyncio.Semaphore(3)

    # Create tasks for all test cases
    tasks = [run_test_case(tc, semaphore) for tc in subset_to_run]

    # Execute all tasks concurrently
    results = await asyncio.gather(*tasks)

    # Convert to Pandas DataFrame
    df_results = pd.DataFrame(results)

    # Save to CSV
    output_filename = "adk_batch_evaluation_report.csv"
    df_results.to_csv(output_filename, index=False)

    print(f"\nEvaluation Complete! Results saved to '{output_filename}'.")

    # Display summary statistics
    print("\n--- Evaluation Summary ---")
    print(f"Average Faithfulness: {df_results['faithfulness'].mean():.2f} / 5.0")
    print(f"Average Completeness: {df_results['completeness'].mean():.2f} / 5.0")

    # Display the first few rows in Colab
    return df_results

In [ ]:
# =====================================================================
# @title ### 5. Entry Point for Colab
# =====================================================================
# In Colab, the event loop is already running. We can await the main function directly.
# If running outside a notebook, use: asyncio.run(main())

df_report = await main() # -1 by default to use all questions

In [ ]:
# @title ### 6. Show results
df_report.head()

# Debugging codes

In [ ]:
# @title ### Inspect Agent Schema (Metadata)
import json
import requests

# Usamos la misma ruta, pero SIN el ":query" al final
AGENT_RESOURCE_NAME = "projects/55231573316/locations/us-central1/reasoningEngines/9130035594284498944"
METADATA_ENDPOINT = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{AGENT_RESOURCE_NAME}"

print("Consultando el esquema del Agente...\n")

try:
    response = requests.get(METADATA_ENDPOINT, headers=HEADERS)
    response.raise_for_status()
    metadata = response.json()

    # Buscamos la definición de la clase y sus métodos (normalmente bajo 'spec' o 'classMethods')
    print("=== METADATA COMPLETA DEL AGENTE ===")
    print(json.dumps(metadata, indent=2))

except Exception as e:
    print(f"Error al obtener metadata: {e}")

In [ ]:
# @title ### 🕵️‍♂️ AGENT PUBLIC METHODS AUDIT
import requests
import json

# Configuration
LOCATION = "us-central1"
AGENT_RESOURCE_NAME = "projects/55231573316/locations/us-central1/reasoningEngines/9130035594284498944"

# We use a GET request to the base resource URL to fetch the deployment metadata
AUDIT_ENDPOINT = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{AGENT_RESOURCE_NAME}"

def audit_agent_deployment():
    print(f"🔍 Auditing Deployment: {AGENT_RESOURCE_NAME}\n")

    try:
        # Fetching the metadata
        response = requests.get(AUDIT_ENDPOINT, headers=HEADERS)
        response.raise_for_status()
        metadata = response.json()

        # Extracting the exposed methods from the spec
        spec = metadata.get('spec', {})
        class_methods = spec.get('classMethods', [])

        if not class_methods:
            print("❌ CRITICAL: No public methods found in the deployment spec.")
            return

        exposed_method_names = [method.get('name') for method in class_methods]

        print("📋 EXPOSED PUBLIC METHODS FOUND:")
        for name in exposed_method_names:
            print(f"  - {name}")

        print("\n" + "="*50)
        print("🎯 VALIDATION RESULT")
        print("="*50)

        # Validating if the core interaction methods exist
        core_methods = ['query', 'stream_query', 'async_stream_query']
        found_core = [m for m in core_methods if m in exposed_method_names]

        if found_core:
            print(f"✅ SUCCESS: Core interaction methods are exposed: {found_core}")
            print("💡 The agent is properly deployed and ready for testing.")
        else:
            print(f"❌ FAIL: Core interaction methods ({core_methods}) are MISSING.")
            print("💡 BLOCKER CONFIRMED: The deployment is missing the interaction entrypoints.")
            print("   The agent can only manage sessions, not answer queries via REST API.")

    except Exception as e:
        print(f"Error during audit: {e}")

# Run the audit
audit_agent_deployment()